# SDSU Climate Informatics Lab

This notebook is a sample tutorial for using AWS Elastic Compute 2 to create AI weather predictions using NVIDIA's FourCastNetv2 Spherical Fourier Neural Operator model and a sample training tutorial for the original FourCastNet model. 

SCIL provides this user guide which will be primarily useful to research groups and weather enthusiasts who would like to produce high quality global weather forecasts and do not have access to a local or shared GPU cluster. Our approach is then to determine the computational resources required and services which can be provided by cloud computing platforms such as Amazon Web Services.

### Getting Started

1) Follow the README.md file in the github repository https://github.com/ikhadir/FCN4AWS. This will guide you through step by step instructions on how to launch an EC2 instance, install PyTorch and CUDA, setup an account to request climate data using CDS API, install the ECMWF ```ai-models``` plugin, and generate a 6-hourly 40 step forecast of 72 variables.
2) Use the ```fcnv2_setup.sh``` script to:

    a) Create ```fourcastnetv2``` directory.
    
    b) Create a ```.cdsapirc``` file for pulling CDS ERA5 data.

    c) Download the precomputed normalization statistics for the ERA5 data.

    d) Install ECMWF ```ai-models``` plugin


### Generate a Prediction using ```ai-models```

In [13]:
#import libraries
#set the working directory to access the precomputed normaliation stats

import os
from ai_models_fourcastnetv2 import model as sfno
import h5py
import cfgrib
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import numpy as np
import xarray as xr

os.getcwd()
os.chdir("fourcastnetv2")


'/Users/imankhadir/SCIL/FCN4AWS/fourcastnetv2'

In [15]:
#Use the .cdsapi file to pull data from CDS for an initial condition
#40 6-hour time steps will be produced and stored in the file 'fourcastnetv2-small.grib'

!ai-models --input cds --date 20230110 --time 0000 fourcastnetv2-small

2025-08-22 10:11:49,139 INFO Writing results to fourcastnetv2-small.grib
2025-08-22 10:11:49,139 INFO Loading ./global_means.npy
2025-08-22 10:11:49,141 INFO Loading ./global_stds.npy
2025-08-22 10:11:49,141 INFO Loading surface fields from CDS
2025-08-22 10:11:50,081 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-08-22 10:11:50,081 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-08-22 10:11:50,671 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-08-22 10:11:50,671 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-08-22 10:11:51,450 INFO Request ID is 7c68442a-10b3-44d6-804f-28b7a61d7328
2025-08-22 10:11:51,450 INFO Request ID is 7c68442a-10b3-44d6-804f-28b7a61d7328
2025-08

### Visualize the Predictions

In [1]:
# Path to your GRIB file, typically a fourcastnetv2 prediction
grib_file = "fourcastnetv2-small.grib"

# Load the GRIB file using xarray
ds = xr.open_dataset(grib_file, engine='cfgrib')

# Display the dataset structure
print(ds)

# Check the shape of a specific variable
for var in ds.data_vars:
    print(f"Shape of '{var}': {ds[var].shape}")



NameError: name 'xr' is not defined

In [2]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np

variable_info = {
    'u10': {'name': '10m Wind Speed', 'unit': 'm/s'},
    'v10': {'name': '10m Wind Speed', 'unit': 'm/s'},
    'msl': {'name': 'Mean Sea Level Pressure', 'unit': 'hPa'},
    'sp': {'name': 'Surface Pressure', 'unit': 'hPa'},
    't': {'name': 'Temperature', 'unit': 'K'}  # Adjust unit if needed (e.g., °C)
}

def robinson_predictions_plot(predictions, variable, t, huh=False):
    dat = predictions[variable].values[t]
    if huh:
        dat = predictions[variable].values[t][0]
    print(dat.shape)

    if variable == 'sp' or variable == 'msl':
        dat = predictions[variable].values[t]/100
    
    # Create subplots with the Robinson projection centered on the Pacific (central_longitude=180)
    central_longitude = 180
    projection = ccrs.Robinson(central_longitude=central_longitude)
    
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8, 5), subplot_kw={'projection': projection})
    
    # Define the extent of the map (in degrees)
    extent = (-180, 180, -90, 90)
    
    # Define the color limits
    vmin, vmax = -2, 1
    
    
    # Plot the prediction data
    ax.set_global()
    im1 = ax.imshow(np.roll(dat, shift=dat.shape[-1]//2, axis=-1), 
                       transform=ccrs.PlateCarree(central_longitude=0), 
                       cmap="jet", extent=extent, origin='upper')#, vmin=vmin, vmax=vmax)
    ax.coastlines()
    ax.set_title(f"FourCastNetv2 {t * 6} Hour Prediction of {variable} from Z:00:00", fontsize=16)
    
    var_name = variable_info.get(variable, {}).get('name', variable)
    var_unit = variable_info.get(variable, {}).get('unit', '')

    # Add colorbar with dynamic label
    cbar = fig.colorbar(im1, ax=ax, orientation='horizontal', fraction=0.046, pad=0.08)
    cbar.set_label(f"{var_name} ({var_unit})", fontsize=12)
    
    for i in np.arange(2):
        gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False)
        gl.top_labels = False
        gl.right_labels = False

    plt.savefig(f'FourCastNetv2_prediction_{variable}_Dec_25_2024_{t * 6}_hours_from_initialization.png')
    fig.tight_layout()
    plt.show()

In [ ]:
#produce maps of 4 different variables 24hr from IC

robinson_predictions_plot(ds, 'u10', 4)
robinson_predictions_plot(ds, 'msl', 4)
robinson_predictions_plot(ds, 'sp', 4)
robinson_predictions_plot(ds, 't', 4, True)